# HLTV scrape

In [ ]:
# Import

import cloudscraper
import pandas as pd
from bs4 import BeautifulSoup
import time
from datetime import datetime
import requests

In [ ]:
# Scrape results

# Cloudscraper automatikusan kezeli a Cloudflare védelmet
scraper = cloudscraper.create_scraper()

url = "https://www.hltv.org/results"
response = scraper.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Parse matches
    matches = soup.find_all('div', class_='result-con')
    
    match_data = []
    for match in matches:
        try:
            # Csapatnevek - a 'team' class-t keresük
            teams = match.find_all('div', class_='team')
            team1 = teams[0].get_text(strip=True) if len(teams) > 0 else "N/A"
            team2 = teams[1].get_text(strip=True) if len(teams) > 1 else "N/A"
            
            # Eredmény - a result-score cellában
            score_element = match.find('td', class_='result-score')
            if score_element:
                score_spans = score_element.find_all('span')
                if len(score_spans) >= 2:
                    score = f"{score_spans[0].get_text(strip=True)}-{score_spans[1].get_text(strip=True)}"
                else:
                    score = score_element.get_text(strip=True)
            else:
                score = "N/A"
            
            # Esemény név
            event_element = match.find('span', class_='event-name')
            event = event_element.get_text(strip=True) if event_element else "N/A"
            
            # Match link
            match_link = match.find('a', class_='a-reset')
            match_url = f"https://www.hltv.org{match_link['href']}" if match_link and match_link.get('href') else "N/A"
            
            match_data.append({
                'team1': team1,
                'team2': team2,
                'score': score,
                'event': event,
                'match_url': match_url
            })
            
        except Exception as e:
            print(f"Error parsing match: {e}")
            continue
    
    print(f"Found {len(match_data)} matches")
    if match_data:
        pd.DataFrame(match_data).to_csv('hltv_results.csv', index=False)
        print("Data saved to hltv_results.csv")
    else:
        print("No match data found")
        
else:
    print(f"Failed to fetch page: {response.status_code}")

In [ ]:
# Rankings

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime

def scrape_team_rankings():
    driver = webdriver.Chrome()
    driver.get("https://www.hltv.org/ranking/teams/")
    
    # Várj, amíg betölt a lista
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ranked-team.standard-box")))
    
    rankings = []
    rank_divs = driver.find_elements(By.CSS_SELECTOR, ".ranked-team.standard-box")
    
    for rank_div in rank_divs:
        try:
            rank = rank_div.find_element(By.CLASS_NAME, "position").text
            team_name = rank_div.find_element(By.CLASS_NAME, "name").text
            points = rank_div.find_element(By.CLASS_NAME, "points").text.replace('(', '').replace(')', '').replace(" HLTV points", "")
            
            team_link = rank_div.find_element(By.TAG_NAME, "a").get_attribute("href")
            team_id = team_link.split('/')[-2]
            profile_link = rank_div.find_element(By.CLASS_NAME, "moreLink").get_attribute("href")
            
            rankings.append({
                'date': datetime.now().strftime('%Y-%m-%d'),
                'rank': int(rank.replace('#', '')),
                'team_id': team_id,
                'team_name': team_name,
                'points': int(points),
                'profile_link': profile_link
            })
        except Exception as e:
            print(f"Hiba: {e}")
            continue
    
    driver.quit()
    return pd.DataFrame(rankings)

# Futtasd hetente → time-series ranking data
rankings = scrape_team_rankings()
display(rankings)

In [ ]:
# Events

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

def scrape_major_events():
    driver = webdriver.Chrome()
    driver.get("https://www.hltv.org/events/archive?eventType=MAJOR")
    
    # Várunk, amíg betöltődnek az események hónapjai
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "events-month")))
    
    events_data = []
    months_divs = driver.find_elements(By.CLASS_NAME, "events-month")
    
    for month_div in months_divs:
        try:
            # Hónap
            month_name = month_div.find_element(By.CLASS_NAME, "standard-headline").text.strip()
            
            # Minden esemény az adott hónapban
            event_links = month_div.find_elements(By.CSS_SELECTOR, "a.small-event.standard-box")
            
            for event in event_links:
                try:
                    event_name = event.find_element(By.CSS_SELECTOR, ".event-col .text-ellipsis").text.strip()
                    team_count = event.find_elements(By.CSS_SELECTOR, ".table tr:first-child td.small-col")[0].text.strip()
                    prize = event.find_elements(By.CSS_SELECTOR, ".table tr:first-child td.prizePoolEllipsis")[0].get_attribute("title").strip()
                    link = event.get_attribute("href").strip()
                    event_id = link.split('/')[4]
                    
                    events_data.append({
                        "month": month_name,
                        "event_name": event_name,
                        "event_id": event_id,
                        "teams": team_count,
                        "prize": prize,
                        "link": link
                    })
                except Exception as e_event:
                    print(f"Esemény hiba: {e_event}")
                    continue
        except Exception as e_month:
            print(f"Hónap hiba: {e_month}")
            continue
    
    driver.quit()
    return pd.DataFrame(events_data)

# Futtatás
major_events = scrape_major_events()
display(major_events.sample(5))

In [ ]:
# Matches of event

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

def scrape_event_results(event_id):
    url = f"https://www.hltv.org/results?event={event_id}"
    driver = webdriver.Chrome()
    driver.get(url)
    
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "results-sublist")))
    
    results_data = []
    sublists = driver.find_elements(By.CLASS_NAME, "results-sublist")
    
    for sublist in sublists:
        try:
            # Mérkőzés dátuma
            match_date = sublist.find_element(By.CLASS_NAME, "standard-headline").text.strip()
            
            # Az összes mérkőzés az adott dátumban
            matches = sublist.find_elements(By.CSS_SELECTOR, ".result-con a")
            
            for match in matches:
                try:
                    link = match.get_attribute("href").strip()
                    
                    team_home = match.find_element(By.CSS_SELECTOR, ".team1 .team").text.strip()
                    team_away = match.find_element(By.CSS_SELECTOR, ".team2 .team").text.strip()
                    
                    # Pontok a sorrend alapján
                    score_spans = match.find_elements(By.CSS_SELECTOR, ".result-score span")
                    score_home = int(score_spans[0].text.strip())
                    score_away = int(score_spans[1].text.strip())
                    
                    map_type = match.find_element(By.CSS_SELECTOR, ".map-and-stars .map-text").text.strip()
                    
                    rounds = int(map_type[-1]) if map_type[:2] == "bo" else 1

                    results_data.append({
                        "date": match_date,
                        "team_home": team_home,
                        "team_away": team_away,
                        "score_home": score_home,
                        "score_away": score_away,
                        "map": map_type,
                        "rounds": rounds,
                        "link": link
                    })
                except Exception as e_match:
                    print(f"Mérkőzés hiba: {e_match}")
                    continue
        except Exception as e_sublist:
            print(f"Dátum hiba: {e_sublist}")
            continue
    
    driver.quit()
    return pd.DataFrame(results_data)

# Példa
event_results = scrape_event_results(7902)
display(event_results.sample(5))


In [ ]:
# Head-to-Head + Match Stats Scraper (pure Selenium)

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import numpy as np
import time

def scrape_head_to_head_and_stats(match_url):
    driver = webdriver.Chrome()
    wait = WebDriverWait(driver, 10)

    driver.get(match_url)
    wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "head-to-head")))

    data = []

    try:
        # --- Head-to-head blokk ---
        h2h_box = driver.find_element(By.CLASS_NAME, "head-to-head")
        container = h2h_box.find_element(By.CLASS_NAME, "standard-box")

        # Csapatnevek
        team1 = container.find_element(By.CSS_SELECTOR, ".team1 .teamName").text.strip()
        team2 = container.find_element(By.CSS_SELECTOR, ".team2 .teamName").text.strip()

        # Wins / Overtimes / Wins
        stats = container.find_elements(By.CSS_SELECTOR, ".flexbox-column.grow .bold")
        wins_team1 = int(stats[0].text.strip())
        overtimes = int(stats[1].text.strip())
        wins_team2 = int(stats[2].text.strip())

        total_non_ot = wins_team1 + wins_team2
        home_win_rate = wins_team1 / total_non_ot if total_non_ot > 0 else None

        # --- Match stat linkek keresése ---
        stat_links = [a.get_attribute("href") for a in driver.find_elements(By.CSS_SELECTOR, "a[href*='hltv.org/stats/matches']")]

        # --- Stat gyűjtés ---
        all_home_ratings, all_away_ratings = [], []
        all_home_adrs, all_away_adrs = [], []
        all_home_swings, all_away_swings = [], []

        for link in stat_links:
            try:
                driver.get(link)
                wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table.totalstats")))
                time.sleep(1.5)

                tables = driver.find_elements(By.CSS_SELECTOR, "table.totalstats")
                print(tables)
                if len(tables) < 2:
                    continue

                # --- Home csapat stat táblázat ---
                home_rows = tables[0].find_elements(By.CSS_SELECTOR, "tr:not(.header-row)")
                for row in home_rows:
                    try:
                        rating = float(row.find_element(By.CSS_SELECTOR, "td.rating").text.strip())
                        adr = float(row.find_element(By.CSS_SELECTOR, "td.adr").text.strip())
                        swing_text = row.find_element(By.CSS_SELECTOR, "td.roundSwing").text.strip().replace("%", "")
                        swing = float(swing_text) if swing_text else None
                        all_home_ratings.append(rating)
                        all_home_adrs.append(adr)
                        if swing is not None:
                            all_home_swings.append(swing)
                    except:
                        continue

                # --- Away csapat stat táblázat ---
                away_rows = tables[1].find_elements(By.CSS_SELECTOR, "tr:not(.header-row)")
                for row in away_rows:
                    try:
                        rating = float(row.find_element(By.CSS_SELECTOR, "td.rating").text.strip())
                        adr = float(row.find_element(By.CSS_SELECTOR, "td.adr").text.strip())
                        swing_text = row.find_element(By.CSS_SELECTOR, "td.roundSwing").text.strip().replace("%", "")
                        swing = float(swing_text) if swing_text else None
                        all_away_ratings.append(rating)
                        all_away_adrs.append(adr)
                        if swing is not None:
                            all_away_swings.append(swing)
                    except:
                        continue

            except Exception as e_stat:
                print(f"⚠️ Hiba stat link feldolgozásakor: {e_stat}")
                continue

        # --- Aggregált statisztikák ---
        def safe_mean(x): return round(np.mean(x), 3) if x else None
        def safe_std(x): return round(np.std(x), 3) if x else None

        team_stats = {
            "home_team": team1,
            "away_team": team2,
            "wins_home": wins_team1,
            "wins_away": wins_team2,
            "overtimes": overtimes,
            "total_non_overtime": total_non_ot,
            "home_win_rate": round(home_win_rate, 4) if home_win_rate else None,

            # --- Home aggregált ---
            "home_team_avg_rating": safe_mean(all_home_ratings),
            "home_team_std_rating": safe_std(all_home_ratings),
            "home_team_avg_ADR": safe_mean(all_home_adrs),
            "home_team_std_ADR": safe_std(all_home_adrs),
            "home_team_avg_Swing": safe_mean(all_home_swings),
            "home_team_std_Swing": safe_std(all_home_swings),

            # --- Away aggregált ---
            "away_team_avg_rating": safe_mean(all_away_ratings),
            "away_team_std_rating": safe_std(all_away_ratings),
            "away_team_avg_ADR": safe_mean(all_away_adrs),
            "away_team_std_ADR": safe_std(all_away_adrs),
            "away_team_avg_Swing": safe_mean(all_away_swings),
            "away_team_std_Swing": safe_std(all_away_swings),

            "source_url": match_url
        }

        data.append(team_stats)

    except Exception as e:
        print(f"❌ Hiba a head-to-head + stat scrape során: {e}")

    driver.quit()
    return pd.DataFrame(data)


# --- Példa futtatás ---
url = "https://www.hltv.org/matches/2382612/virtuspro-vs-pain-blasttv-austin-major-2025"
df = scrape_head_to_head_and_stats(url)
display(df)


In [ ]:
# Match stat links

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def scrape_stats_links(match_url):
    driver = webdriver.Chrome()
    driver.get(match_url)
    
    # Várj, amíg betöltődik a fő tartalom
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.TAG_NAME, "a")))
    
    all_links = driver.find_elements(By.TAG_NAME, "a")
    stats_links = []

    for link in all_links:
        href = link.get_attribute("href")
        if href and "hltv.org/stats/" in href:
            stats_links.append(href)
    
    driver.quit()
    return stats_links

# Példa futtatás
match_url = "https://www.hltv.org/matches/2382616/natus-vincere-vs-vitality-blasttv-austin-major-2025"
stats_links = scrape_stats_links(match_url)
for statlink in stats_links:
    print(statlink)

# Esports Devs API

In [ ]:
# Find leaf keys function

from typing import Any, List, Tuple

def _contains_dict(obj: Any) -> bool:
    """Visszaadja True-t, ha obj maga dict, vagy (rekurzívan) tartalmaz dict-et (pl. listában)."""
    if isinstance(obj, dict):
        return True
    if isinstance(obj, list):
        for el in obj:
            if _contains_dict(el):
                return True
    return False

def find_leaf_keys(data: dict, *, sep: str = ".") -> List[str]:
    """
    Visszaadja a levélkulcsok teljes útvonalait (pl. "a.b.c") azoknak a kulcsoknak,
    amelyek értéke nem dict és nem tartalmaz dict-et listában sem.
    """
    leaves: List[str] = []

    def _recurse(obj: Any, path: List[str]):
        if isinstance(obj, dict):
            for k, v in obj.items():
                new_path = path + [str(k)]
                # ha az érték dict-et (vagy listában dict-et) tartalmaz -> megyünk tovább
                if _contains_dict(v):
                    _recurse(v, new_path)
                else:
                    # ez levél: nem dict és a list sem tartalmaz dict-et
                    leaves.append(sep.join(new_path))
        elif isinstance(obj, list):
            # listát akkor járjuk be, ha szeretnénk megtalálni a benne lévő dict-ek leveleit
            for idx, el in enumerate(obj):
                _recurse(el, path + [f"[{idx}]"])
        else:
            # objektum önmagában (nem dict, nem list): ha path utolsó elemre vonatkozik, már lefutott korábban
            pass

    _recurse(data, [])
    return leaves

# Ha csak a levélkulcs "név"-eket akarod (nem teljes útvonal), ezt használhatod:
def find_leaf_key_names(data: dict) -> List[str]:
    paths = find_leaf_keys(data)
    return [p.split(".")[-1] for p in paths]

In [ ]:
import requests
import os 
import dotenv

dotenv.load_dotenv()

API_KEY = os.getenv('API_KEY')
BASE_URL = "https://esports-devs.p.rapidapi.com"

headers = {
    "x-rapidapi-key": API_KEY,
    "x-rapidapi-host": "esports-devs.p.rapidapi.com"
}

In [ ]:
# Leagues

response = requests.get(
    f"{BASE_URL}/leagues",
    headers=headers,
    params={
        "limit": 10,
        "offset": 0,
        "class_id": "eq.112"
    }
)

leagues = response.json()
print(leagues)

In [ ]:
for league in leagues:
    print(f"{league['name']} ({league['id']})")

In [ ]:
# Tournaments

response = requests.get(
    f"{BASE_URL}/tournaments",
    headers=headers,
    params={
        "league_id": "eq.2175",  # BLAST Premier
        "limit": 10
    }
)
tournaments = response.json()
print(tournaments)

In [ ]:
for tour in tournaments:
    print(f"{tour['name']} ({tour['id']})")

In [ ]:
# Seasons

response = requests.get(
    f"{BASE_URL}/seasons",
    headers=headers,
    params={
        "league_id": "eq.2191",
        "limit": 10
    }
)

seasons = response.json()
print(seasons)

In [ ]:
for season in seasons:
    print(f"{season['name']} ({season['id']})")

In [ ]:
# Matches by season

response = requests.get(
    f"{BASE_URL}/matches",
    headers=headers,
    params={
        "limit": 10,
        "offset": 0,
        "tournament_id": "eq.16810"
        #"season_id": "eq.13044"
    }
)

matches = response.json()
print(matches)

In [ ]:
for key in find_leaf_keys(matches[0]):
    print(key)

In [ ]:
for match in matches:
    print(match['start_time'])
    print(f"{match['home_team_name']} ({match['home_team_id']})" \
          f" vs {match['away_team_name']} ({match['away_team_id']})" \
          f" (match: {match['id']})")
    print(f"{match['home_team_score']['current']}" \
          f" - {match['away_team_score']['current']}\n")

In [ ]:
# Games of match

response = requests.get(
    f"{BASE_URL}/matches-games",
    headers=headers,
    params={
        "limit": 10,
        "offset": 0,
        "match_id": "eq.88423"
    }
)

games = response.json()
print(games)

In [ ]:
for key in find_leaf_keys(games[0]):
    print(key)

In [ ]:
for game in games:
    print(f"{game['map']} ({game['id']})\n" \
          f"{game['home_team_score']['display']} - {game['away_team_score']['display']}\n" \
          f"- Stats: {game['has_statistics']}\n- Rounds: {game['has_rounds']}\n- Lineups: {game['has_lineups']}\n")

In [ ]:
# Team

response = requests.get(
    f"{BASE_URL}/teams",
    headers=headers,
    params={
        "limit": 50,
        "id": "eq.2433"
    }
)

teams = response.json()
print(teams)

In [ ]:
# Odds coverage

response = requests.get(
    f"{BASE_URL}/odds/coverage",
    headers=headers,
    params={
        "limit": 50,
        "match_id": "eq.222076"
    }
)

odds_cov = response.json()
print(odds_cov)

In [ ]:
import pandas as pd

leagues = pd.read_csv("output/leagues.csv")
seasons = pd.read_csv("output/seasons.csv")
matches = pd.read_csv("output/matches.csv")

#display(leagues) # 2104 - blast premier
#display(seasons[seasons.league_id == 2104]) # 22938 - globals final 2020
display(matches[matches.season_id == 22938])

In [ ]:
print(leagues.name.unique())